# OCR and field-extraction benchmark

Which pipeline should Draftly use to read scanned Sri Lankan conveyancing documents?
This notebook answers four separate questions on one shared test set:

1. Which engine reads the text most accurately?
2. Which pipeline extracts critical legal fields most accurately?
3. Does preprocessing or cropping actually improve results?
4. Can the system show where every value came from?

**This notebook only reads results.** It never calls a model, so it is free to
re-run. Produce results first:

```powershell
uv run python ocr-benchmark/runner.py stub          # offline, free, proves the harness
uv run python ocr-benchmark/runner.py A B D         # the real comparison
uv run python ocr-benchmark/score.py                # scores whatever has been run
```

## Two rules

**Clear all outputs before committing.** The provenance previews render real client
pages — names, NICs, addresses. A committed notebook with those outputs in it is
exactly the leak the case manifests forbid.

**Critical identifiers are scored exact-match only.** `00030085091` against
`00030085090` is a failure no matter how similar the strings are, because the legal
result is completely wrong.

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "ocr-benchmark" else Path.cwd() / "ocr-benchmark"))

import matplotlib.pyplot as plt
import pandas as pd

import config, engines, score, viz

viz.use_style()
pd.set_option("display.max_rows", 200)

print("inputs :", config.INPUTS)
print("runs   :", config.RUNS)
for engine, state in engines.capabilities().items():
    print(f"  {engine:10s} {state}")
labels_present = any((config.LABELS / kind).glob("*.json") for kind in ("fields", "ocr-region", "routing") if (config.LABELS / kind).is_dir())
print("  labels     ", "present" if labels_present else "none yet (CER/WER unavailable)")

In [ ]:
# Score everything that has been run, then load the aggregate report.
score.main()

summary = json.loads((config.REPORTS / "metrics.json").read_text(encoding="utf-8"))
metrics = summary["runs"]

rows = []
for path in sorted(config.RUNS.glob("*/predictions.csv")):
    # dtype=str and keep_default_na=False: "0021" must not become 21, and an
    # empty expected value must not become NaN.
    rows.append(pd.read_csv(path, dtype=str, keep_default_na=False))
fields_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print(f"{len(metrics)} runs, {len(fields_df)} scored field rows")

In [ ]:
# Run table — the reproducibility record for every result below.
manifest_rows = []
for path in sorted(config.RUNS.glob("*/config.json")):
    m = json.loads(path.read_text(encoding="utf-8"))
    manifest_rows.append({
        "variant": m["variantId"], "pipeline": m["pipeline"], "status": m["status"],
        "reason": m.get("reason", ""), "dpi": m.get("renderDpi"),
        "variantRender": m.get("renderVariant"), "extractModel": (m.get("models") or {}).get("extract"),
        "detector": m.get("detector"), "commit": m.get("codeCommit"),
        "prompt": (m.get("promptVersion") or {}).get("page"),
        "dataset": (m.get("datasetVersion") or "")[:8],
    })
pd.DataFrame(manifest_rows)

## The dataset

In [ ]:
import pypdfium2 as pdfium

expected = score.load_expected()
docs = []
for pdf_path in sorted(config.INPUTS.glob("*.pdf")):
    doc = pdfium.PdfDocument(pdf_path)
    try:
        pages = len(doc)
    finally:
        doc.close()
    kind = expected.get(pdf_path.name, {}).get("kind", "unlabelled")
    docs.append((pdf_path.stem.replace("source-", "")[:34], pages, kind))

fig, ax = plt.subplots(figsize=(10, 4.2))
viz.dataset_pages(ax, docs)
plt.tight_layout(); plt.show()

labelled = sum(len(v.get("fields", {})) for v in expected.values())
print(f"{labelled} labelled fields across {sum(1 for v in expected.values() if v.get('fields'))} documents")
print(f"{len(score.CRITICAL_KEYS)} field keys are marked critical")

## Question 2 — which pipeline extracts critical fields most accurately?

This is the primary Draftly metric, so it comes first.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.55 * max(3, len(metrics)) + 1.6))
viz.critical_accuracy(ax, metrics)
plt.tight_layout(); plt.show()

# Table view, because a chart alone is not an accessible result.
pd.DataFrame([
    {
        "variant": v,
        "status": m["status"],
        "critical": m["criticalFieldAccuracy"]["exactMatch"],
        "critical n": m["criticalFieldAccuracy"]["scored"],
        "all fields (platform norm)": m["fieldAccuracy"]["exactMatchPlatform"],
        "all fields (strict norm)": m["fieldAccuracy"]["exactMatchStrict"],
        "omitted": m["omissionRate"],
        "invented": m["inventedValueRate"],
        "kind acc": m["kindAccuracy"],
        "usd": m["cost"]["estimatedUsd"],
    }
    for v, m in sorted(metrics.items())
])

In [ ]:
records = fields_df.to_dict("records") if not fields_df.empty else []
height = 0.22 * max(8, len({r["key"] for r in records} if records else {1})) + 2
fig, ax = plt.subplots(figsize=(9, height))
viz.field_status_heatmap(ax, records)
plt.tight_layout(); plt.show()

## Question 1 — which engine reads the text most accurately?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
viz.text_error_rates(ax, metrics)
plt.tight_layout(); plt.show()

## Question 3 — does preprocessing or resolution actually help?

```powershell
uv run python ocr-benchmark/runner.py A --ablation dpi
uv run python ocr-benchmark/runner.py B --ablation variant
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
viz.dpi_ablation(axes[0], metrics)
viz.preprocessing_ablation(axes[1], metrics)
plt.tight_layout(); plt.show()

## Cross-run agreement

Agreement needs no ground truth at all, which makes it the fastest way to decide
which pages are worth annotating first. The second chart checks whether agreement
is actually a usable proxy for correctness — if runs agree and are still wrong, the
ensemble's confidence signal cannot be trusted.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
viz.agreement_matrix(ax, summary.get("agreementMatrix", {}))
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 2.6))
viz.agreement_vs_truth(ax, summary.get("agreementVsTruth", {}))
plt.tight_layout(); plt.show()

## Question 4 — provenance, and the risk metrics

For lawyer verification, the correct text from the wrong region is still a failure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
viz.provenance_mix(axes[0], metrics)
viz.risk_rates(axes[1], metrics)
plt.tight_layout(); plt.show()

## Cost and latency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
viz.cost_vs_accuracy(axes[0], metrics)
viz.latency_by_run(axes[1], metrics)
plt.tight_layout(); plt.show()
print("cumulative provider spend:", engines.usage_summary())

## Provenance overlay preview

**Renders real client pages. Clear outputs before committing.**

In [ ]:
import render

SHOW_CLIENT_PAGES = False  # flip to True locally; never commit with outputs

if not SHOW_CLIENT_PAGES:
    print("skipped — set SHOW_CLIENT_PAGES = True to preview boxes on real pages")
else:
    variant = next((v for v, m in metrics.items() if m["status"] == "completed" and m["pages"]), None)
    results = config.RUNS / variant.replace("@", "_at_").replace("=", "-") / "results.jsonl"
    docs_out = [json.loads(line) for line in results.read_text(encoding="utf-8").splitlines() if line.strip()]
    shown = 0
    for doc in docs_out:
        pdf_path = config.INPUTS / doc["doc_id"]
        if not pdf_path.is_file() or shown >= 2:
            continue
        page_no = next((f["page_no"] for f in doc["fields"] if f.get("bbox")), None)
        if page_no is None:
            continue
        pages = render.render_doc(pdf_path, dpi=200, variant="original", page_limit=page_no)
        page = pages[page_no - 1]
        boxed = [f for f in doc["fields"] if f.get("page_no") == page_no and f.get("bbox")]
        fig, ax = plt.subplots(figsize=(8, 11))
        viz.overlay_boxes(ax, page.image, boxed, title=f"{variant} — {doc['doc_id'][:40]} p{page_no}")
        plt.tight_layout(); plt.show()
        shown += 1

## Error triage

Every mismatch, for manual inspection. Fix the pipeline from what you see here —
never edit the expectations to match the model.

In [ ]:
if fields_df.empty:
    print("no scored fields yet")
else:
    bad = fields_df[fields_df["outcome"].isin(["wrong", "missing"])]
    print(f"{len(bad)} mismatches. Per-run detail: runs/<variant>/mismatches.csv (gitignored)")
    display(bad[["variant_id", "doc_id", "key", "critical", "outcome", "page", "provenance", "engine"]].head(60))
    print("\nmost-failed fields:")
    display(bad.groupby("key").size().sort_values(ascending=False).head(15))

## Decision rule

Do not pick the winner on overall CER. Pick the configuration with the best
combination of:

1. critical-field exact-match accuracy
2. lowest invented-value rate
3. correct provenance
4. acceptable manual-review rate
5. acceptable cost and latency

### What is not yet measurable

- **CER and WER by language and text type** need human region transcripts under
  `ocr-benchmark/labels/ocr-region/`. Three hand-typed pages would make it real.
- **Bounding-box IoU** needs annotated boxes. Until then provenance is measured by
  plausibility: does the returned box contain the returned value, and is the value
  verbatim present in the page text.
- **Runs C, E and F** need a box-producing engine. Surya is GPL-3.0, which the
  platform's own AGPL rejection of PyMuPDF suggests needs a licence ruling;
  `rapidocr` is Apache-2.0 and already installed as the fallback detector.
- **A 26-page smoke set decides nothing on its own.** The Wilson intervals above
  are wide by design. Expanding to 100–200 pages across multiple matters, split by
  matter rather than by page, is what turns this into a decision.